<a href="https://colab.research.google.com/github/Lekanggy/MLrepo/blob/main/U_Net_Arch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
import warnings
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (Input, Conv2D, MaxPooling2D, Conv2DTranspose, concatenate)

import numpy as np
import matplotlib.pyplot as plt
import os

In [15]:
def dice_coef(y_true, y_pred, smooth=1e-6):
  y_true_f = tf.keras.backend.flatten(y_true)
  y_pred_f = tf.keras.backend.flatten(y_pred)

  intersection = tf.reduce_sum(y_true_f * y_pred_f)
  return (2 * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)


def jaccard_index(y_true, y_pred, smooth=1e-6):
  y_true_f = tf.keras.backend.flatten(y_true)
  y_pred_f = tf.keras.backend.flatten(y_pred)
  intersection = tf.reduce_sum(y_true_f * y_pred_f)
  union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
  return (intersection + smooth) / (union + smooth)

In [8]:
y_true_sample = tf.constant([[0, 1, 0], [1, 1, 0], [0, 0, 1]], dtype=tf.float32)
y_pred_sample = tf.constant([[0, 1, 1], [1, 1, 0], [0, 1, 1]], dtype=tf.float32)

print("Sample y_true:")
print(y_true_sample.numpy())
print("\nSample y_pred:")
print(y_pred_sample.numpy())


Sample y_true:
[[0. 1. 0.]
 [1. 1. 0.]
 [0. 0. 1.]]

Sample y_pred:
[[0. 1. 1.]
 [1. 1. 0.]
 [0. 1. 1.]]


In [13]:
dice_coef(y_true_sample, y_pred_sample)

<tf.Tensor: shape=(), dtype=float32, numpy=0.800000011920929>

In [16]:
jaccard_index(y_true_sample, y_pred_sample)

<tf.Tensor: shape=(), dtype=float32, numpy=0.6666667461395264>

In [17]:
def unet_arch(input_shape=(128, 128, 1)):
  inputs = Input(input_shape)

  #Down Sampling Technique
  c1 = Conv2D(64, 3, activation='relu', padding='same')(inputs)
  c1 = Conv2D(64, 3, activation='relu', padding='same')(c1)
  p1 = MaxPooling2D(2)(c1)

  c2 = Conv2D(128, 3, activation='relu', padding='same')(p1)
  c2 = Conv2D(128, 3, activation='relu', padding='same')(c2)
  p2 = MaxPooling2D(2)(c2)

  c3 = Conv2D(256, 3, activation='relu', padding='same')(p2)
  C3 = Conv2D(256, 3, activation='relu', padding='same')(c3)
  p3 = MaxPooling2D(2)(c3)

  c4 = Conv2D(512, 3, activation='relu', padding='same')(p3)
  c4 = Conv2D(512, 3, activation='relu', padding='same')(c4)
  p4 = MaxPooling2D(2)(c4)

  # Flat layer of the model
  c5 = Conv2D(1024, 3, activation='relu', padding='same')(p4)
  c5 = Conv2D(1024, 3, activation='relu', padding='same')(c5)

  #Upsampling begin
  u6 = Conv2DTranspose(512, 3, strides=2, padding='same')(c5)
  u6 = concatenate([u6, c4]) # Skip connection is establish here
  c6 = Conv2D(512, 3, activation='relu', padding='same')(u6)
  c6 = Conv2D(512, 3, activation='relu', padding='same')(c6)

  u7 = Conv2DTranspose(256, 3, strides=2, padding='same')(c6)
  u7 = concatenate([u7, c3])
  c7 = Conv2D(256, 3, activation='relu', padding='same')(u7)
  c7 = Conv2D(256, 3, activation='relu', padding='same')(c7)

  u8 = Conv2DTranspose(128, 3, strides=2, padding='same')(c7)
  u8 = concatenate([u8, c2]) # Skip connection is establish here
  c8 = Conv2D(128, 3, activation='relu', padding='same')(u8)
  c8 = Conv2D(128, 3, activation='relu', padding='same')(c8)

  u9 = Conv2DTranspose(64, 3, strides=2, padding='same')(c8)
  u9 = concatenate([u9, c1])
  c9 = Conv2D(64, 3, activation='relu', padding='same')(u9)
  c9 = Conv2D(64, 3, activation='relu', padding='same')(c9)

  outputs = Conv2D(1, 1, activation='sigmoid')(c9)
  model = Model(inputs, outputs)

  return model




In [19]:
model = unet_arch()

model.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
              loss=tf.keras.losses.BinaryCrossentropy(),
              metrics=[dice_coef,jaccard_index])

model.summary()

#Each parameter is getting from the following formular
#params = (kh * kw * cin + 1) * cout
#e.g kh = 3, kw = 3, cin = 1, cout = 64
#(3 * 3 * 1 + 1) * 64 = 10 * 64 = 640

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 128, 128,  │        640 │ input_layer_1[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_20 (Conv2D)  │ (None, 128, 128,  │     36,928 │ conv2d_19[0][0]   │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_4     │ (None, 64, 64,    │          0 │ conv2d_20[0][0]   │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_21 (Conv2D)  │ (None, 64, 64,    │     73,856 │ max_pooling2d_4[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_22 (Conv2D)  │ (None, 64, 64,    │    147,584 │ conv2d_21[0][0]   │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_5     │ (None, 32, 32,    │          0 │ conv2d_22[0][0]   │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_23 (Conv2D)  │ (None, 32, 32,    │    295,168 │ max_pooling2d_5[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_6     │ (None, 16, 16,    │          0 │ conv2d_23[0][0]   │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_25 (Conv2D)  │ (None, 16, 16,    │  1,180,160 │ max_pooling2d_6[… │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_26 (Conv2D)  │ (None, 16, 16,    │  2,359,808 │ conv2d_25[0][0]   │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_7     │ (None, 8, 8, 512) │          0 │ conv2d_26[0][0]   │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_27 (Conv2D)  │ (None, 8, 8,      │  4,719,616 │ max_pooling2d_7[… │
│                     │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_28 (Conv2D)  │ (None, 8, 8,      │  9,438,208 │ conv2d_27[0][0]   │
│                     │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose_4  │ (None, 16, 16,    │  4,719,104 │ conv2d_28[0][0]   │
│ (Conv2DTranspose)   │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_4       │ (None, 16, 16,    │          0 │ conv2d_transpose… │
│ (Concatenate)       │ 1024)             │            │ conv2d_26[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_29 (Conv2D)  │ (None, 16, 16,    │  4,719,104 │ concatenate_4[0]

 Total params: 33,922,113 (129.40 MB)

 Trainable params: 33,922,113 (129.40 MB)

 Non-trainable params: 0 (0.00 B)